# Multiverse of Monsoon — GRAST GPU Training (Kaggle)

Self-contained training notebook for **GRAST** (and `FallbackCNN`).

### Before clicking **Run All**:
1. Attach a Kaggle Dataset containing `india_climate_train.nc` and `stats.json` (the notebook auto-discovers it under `/kaggle/input`).
2. In Kaggle **Settings -> Accelerator**, select **GPU T4 x2** or **GPU P100**.
3. After training finishes, download `/kaggle/working/artifacts/primary_grast.pt` and place it in `artifacts/primary_grast.pt` in your local repo.


In [ ]:
import json
import math
import time
import zlib
from pathlib import Path
from typing import Tuple

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# ===========================================================================
# Configuration
# ===========================================================================
DATA_DIR_OVERRIDE = None   # Set to Path("/kaggle/input/your-dataset") if auto-find fails
OUT_DIR = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CLASS = "primary"    # "primary" (GRAST) or "fallback" (FallbackCNN)
BASE_CH     = 64           # 64 (~1.4M params, great for GPU) or 32 (~355k params)
HORIZON     = 8            # Progressive curriculum target horizon (e.g., 8 or 10)
EPOCHS      = 24           # Total epochs across curriculum phases
BATCH_SIZE  = 32
LR          = 3e-4
DROPOUT     = 0.05
CURRICULUM  = True         # True: 1 -> 2 -> 4 -> HORIZON
SEED        = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def find_data_files() -> Tuple[Path, Path]:
    if DATA_DIR_OVERRIDE is not None:
        d = Path(DATA_DIR_OVERRIDE)
        return d / "india_climate_train.nc", d / "stats.json"
    for root in [Path("/kaggle/input"), Path("data"), Path(".")]:
        if root.exists():
            nc_hits = list(root.rglob("india_climate_train.nc"))
            if nc_hits:
                nc_p = nc_hits[0]
                st_p = nc_p.parent / "stats.json"
                if st_p.exists():
                    return nc_p, st_p
    raise FileNotFoundError("Could not find india_climate_train.nc and stats.json. Set DATA_DIR_OVERRIDE.")

NC_PATH, STATS_PATH = find_data_files()
print(f"NC_PATH:    {NC_PATH}")
print(f"STATS_PATH: {STATS_PATH}")
print(f"OUT_DIR:    {OUT_DIR}")


In [ ]:
# ===========================================================================
# Data Loading, Normalization, Windowing, and Geo Features
# ===========================================================================
CHANNELS = ["t2m", "tp", "u10", "v10", "msl", "tcwv"]
N_CHANNELS = 6
GRID_H = 32
GRID_W = 36
N_GEO_CHANNELS = 6
N_HISTORY = 3
N_DOY = 2
C_IN = N_HISTORY * N_CHANNELS + N_GEO_CHANNELS + N_DOY  # 26

SPLIT = {
    "train": ("2005-01-01", "2017-12-31"),
    "val":   ("2018-01-01", "2019-12-31"),
    "test":  ("2020-01-01", "2021-12-31"),
}


def _read_dataset_all_chunks(h5file: h5py.File, name: str) -> np.ndarray:
    ds = h5file[name]
    shape = ds.shape
    dtype = ds.dtype
    plist = ds.id.get_create_plist()
    nfilters = plist.get_nfilters()
    if nfilters == 0:
        return ds[()]

    has_deflate = False
    has_shuffle = False
    elem_size = dtype.itemsize
    for i in range(nfilters):
        ftype, _, _, _ = plist.get_filter(i)
        if ftype == h5py.h5z.FILTER_DEFLATE:
            has_deflate = True
        if ftype == h5py.h5z.FILTER_SHUFFLE:
            has_shuffle = True
    if not has_deflate:
        return ds[()]

    out = np.empty(shape, dtype=dtype)
    nc = ds.id.get_num_chunks()
    with open(str(h5file.filename), "rb") as raw_fp:
        for ci in range(nc):
            info = ds.id.get_chunk_info(ci)
            coff = info.chunk_offset
            raw_fp.seek(info.byte_offset)
            compressed = raw_fp.read(info.size)
            raw_bytes = np.frombuffer(zlib.decompress(compressed), dtype=np.uint8)

            valid_shape = tuple(min(cs, s - co) for cs, s, co in zip(ds.chunks, shape, coff))
            full_shape = tuple(ds.chunks)
            n_full = int(np.prod(full_shape))
            if has_shuffle:
                unshuffled = np.empty(n_full * elem_size, dtype=np.uint8)
                for b in range(elem_size):
                    unshuffled[b::elem_size] = raw_bytes[b * n_full : (b + 1) * n_full]
            else:
                unshuffled = raw_bytes

            chunk_arr = np.frombuffer(unshuffled.tobytes(), dtype=dtype).reshape(full_shape)
            chunk_arr = chunk_arr[tuple(slice(0, vs) for vs in valid_shape)]
            out[tuple(slice(co, co + vs) for co, vs in zip(coff, valid_shape))] = chunk_arr
    return out


class DatasetReader:
    def __init__(self, nc_path: Path = NC_PATH) -> None:
        import xarray as xr
        try:
            ds = xr.open_dataset(nc_path, engine="h5netcdf")
            raw = ds["data"].values.astype("float32")
            static = ds["static"].values.astype("float32")
            lats = ds["lat"].values.astype("float32")
            lons = ds["lon"].values.astype("float32")
            dates = ds["time"].values.astype("datetime64[D]")
            ds.close()
        except OSError:
            print("Falling back to chunked HDF5 reader...")
            with h5py.File(nc_path, "r") as f:
                raw = _read_dataset_all_chunks(f, "data").astype("float32")
                static = _read_dataset_all_chunks(f, "static").astype("float32")
                lats = _read_dataset_all_chunks(f, "lat").astype("float32")
                lons = _read_dataset_all_chunks(f, "lon").astype("float32")
            ds_meta = xr.open_dataset(nc_path, engine="h5netcdf")
            dates = ds_meta["time"].values.astype("datetime64[D]")
            ds_meta.close()

        assert raw.shape[1:] == (N_CHANNELS, GRID_H, GRID_W)
        assert np.isfinite(raw).all(), "NaN/Inf in data"
        assert raw[:, 1].min() >= 0, "negative precipitation"
        assert static.shape == (2, GRID_H, GRID_W)

        self.data = raw
        self.static = static
        self.lats = lats
        self.lons = lons
        self._dates = dates

    def split_indices(self, split: str) -> np.ndarray:
        start, end = SPLIT[split]
        s, e = np.datetime64(start, "D"), np.datetime64(end, "D")
        return np.where((self._dates >= s) & (self._dates <= e))[0]

    def doy_sincos(self, day_idx: int) -> np.ndarray:
        t = self._dates[day_idx]
        yr_start = t.astype("datetime64[Y]").astype("datetime64[D]")
        doy = int((t - yr_start).astype(int)) + 1
        angle = 2.0 * np.pi * doy / 365.0
        return np.array([np.sin(angle), np.cos(angle)], dtype="float32")


class Transform:
    def __init__(self, stats_path: Path = STATS_PATH) -> None:
        with open(stats_path) as f:
            s = json.load(f)
        self.mean = np.array(s["mean"], dtype="float32")
        self.std  = np.array(s["std"],  dtype="float32")

    def normalize(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean[:, None, None]) / self.std[:, None, None]

    def denormalize(self, z: np.ndarray) -> np.ndarray:
        return z * self.std[:, None, None] + self.mean[:, None, None]


class WindowIndex:
    def __init__(self, indices: np.ndarray, n_inputs: int = 3, max_horizon: int = 10) -> None:
        total_len = n_inputs + max_horizon
        self.windows = []
        for i in range(len(indices) - total_len + 1):
            block = indices[i : i + total_len]
            if np.all(np.diff(block) == 1):
                self.windows.append((block[:n_inputs], block[n_inputs:]))

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, i: int):
        return self.windows[i]


class ClimateWindowDataset(Dataset):
    def __init__(self, reader: DatasetReader, transform: Transform, window_index: WindowIndex, horizon: int) -> None:
        self.reader = reader
        self.transform = transform
        self.window_index = window_index
        self.horizon = horizon

    def __len__(self) -> int:
        return len(self.window_index)

    def __getitem__(self, idx: int):
        inp_days, tgt_days = self.window_index[idx]
        x_phys = self.reader.data[inp_days]
        x_norm = np.stack([self.transform.normalize(x_phys[i]) for i in range(3)], axis=0)
        h = min(self.horizon, len(tgt_days))
        y_phys = self.reader.data[tgt_days[:h]]
        y_norm = np.stack([self.transform.normalize(y_phys[i]) for i in range(h)], axis=0)
        doy = self.reader.doy_sincos(inp_days[-1])
        return (
            torch.from_numpy(x_norm).float(),
            torch.from_numpy(y_norm).float(),
            torch.tensor(doy[0], dtype=torch.float32),
            torch.tensor(doy[1], dtype=torch.float32),
        )


def build_geo_features(lsm: np.ndarray, orography: np.ndarray, lats: np.ndarray, lons: np.ndarray) -> np.ndarray:
    H, W = lsm.shape
    lsm_ch = lsm.astype("float32")
    oro_min, oro_max = orography.min(), max(orography.max(), 1.0)
    oro_ch = ((orography - oro_min) / (oro_max - oro_min)).astype("float32")
    lat_1d = ((lats - 6.5) / (37.5 - 6.5) * 2 - 1).astype("float32")
    lat_ch = np.broadcast_to(lat_1d[:, None], (H, W)).copy()
    lon_1d = ((lons - 65.5) / (100.5 - 65.5) * 2 - 1).astype("float32")
    lon_ch = np.broadcast_to(lon_1d[None, :], (H, W)).copy()
    cos_lat_1d = np.cos(np.deg2rad(lats)).astype("float32")
    cos_lat_ch = np.broadcast_to(cos_lat_1d[:, None], (H, W)).copy()
    dx_km_1d = (6371.0 * np.cos(np.deg2rad(lats)) * np.deg2rad(1.0)).astype("float32")
    dx_ch = np.broadcast_to((dx_km_1d / dx_km_1d.max())[:, None], (H, W)).copy()
    return np.stack([lsm_ch, oro_ch, lat_ch, lon_ch, cos_lat_ch, dx_ch], axis=0)


In [ ]:
# ===========================================================================
# Model Architecture (GRAST & FallbackCNN — identical to src/models/grast.py)
# ===========================================================================
class NonPeriodicPad(nn.Module):
    def __init__(self, pad: int = 1) -> None:
        super().__init__()
        self.pad = pad

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.pad(x, (self.pad,) * 4, mode="replicate")


class ResBlock(nn.Module):
    def __init__(self, channels: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.pad = NonPeriodicPad(1)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=0, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=0, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.drop  = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.bn1(self.conv1(self.pad(x))), inplace=True)
        h = self.drop(h)
        h = self.bn2(self.conv2(self.pad(h)))
        return F.relu(h + x, inplace=True)


class DownBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.pad  = NonPeriodicPad(1)
        self.conv = nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=0, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.bn(self.conv(self.pad(x))), inplace=True)


class UpBlock(nn.Module):
    def __init__(self, coarse_ch: int, fine_ch: int, out_ch: int) -> None:
        super().__init__()
        self.pad  = NonPeriodicPad(1)
        self.conv = nn.Conv2d(coarse_ch + fine_ch, out_ch, 3, padding=0, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, coarse: torch.Tensor, fine: torch.Tensor) -> torch.Tensor:
        up = F.interpolate(coarse, size=fine.shape[-2:], mode="bilinear", align_corners=False)
        return F.relu(self.bn(self.conv(self.pad(torch.cat([up, fine], dim=1)))), inplace=True)


class GRAST(nn.Module):
    def __init__(self, c_in: int = C_IN, base_ch: int = 64, n_fine_pre: int = 2,
                 n_coarse: int = 3, n_fine_post: int = 2, dropout: float = 0.05) -> None:
        super().__init__()
        coarse_ch = base_ch * 2
        self.encoder = nn.Sequential(
            NonPeriodicPad(1),
            nn.Conv2d(c_in, base_ch, 3, padding=0, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
        )
        self.fine_pre  = nn.Sequential(*[ResBlock(base_ch, dropout) for _ in range(n_fine_pre)])
        self.down      = DownBlock(base_ch, coarse_ch)
        self.coarse    = nn.Sequential(*[ResBlock(coarse_ch, dropout) for _ in range(n_coarse)])
        self.up        = UpBlock(coarse_ch, base_ch, base_ch)
        self.fine_post = nn.Sequential(*[ResBlock(base_ch, dropout) for _ in range(n_fine_post)])
        self.head = nn.Sequential(
            NonPeriodicPad(1),
            nn.Conv2d(base_ch, base_ch // 2, 3, padding=0, bias=True),
            nn.ReLU(inplace=True),
            NonPeriodicPad(1),
            nn.Conv2d(base_ch // 2, N_CHANNELS, 3, padding=0, bias=True),
        )
        self.residual_scale = nn.Parameter(torch.ones(N_CHANNELS) * 0.1)
        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        enc    = self.encoder(x)
        fine0  = self.fine_pre(enc)
        coarse = self.coarse(self.down(fine0))
        merged = self.up(coarse, fine0)
        feat   = self.fine_post(merged)
        dz     = self.head(feat)
        return dz * self.residual_scale.view(1, N_CHANNELS, 1, 1)


class FallbackCNN(nn.Module):
    def __init__(self, c_in: int = C_IN, base_ch: int = 32, n_res: int = 4, dropout: float = 0.0) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            NonPeriodicPad(1),
            nn.Conv2d(c_in, base_ch, 3, padding=0, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
        )
        self.res_blocks = nn.Sequential(*[ResBlock(base_ch, dropout) for _ in range(n_res)])
        self.head = nn.Sequential(
            NonPeriodicPad(1),
            nn.Conv2d(base_ch, N_CHANNELS, 3, padding=0, bias=True),
        )
        self.residual_scale = nn.Parameter(torch.ones(N_CHANNELS) * 0.1)
        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dz = self.head(self.res_blocks(self.encoder(x)))
        return dz * self.residual_scale.view(1, N_CHANNELS, 1, 1)


In [ ]:
# ===========================================================================
# Differentiable Rollout & Multi-Step Huber Loss
# ===========================================================================
RAIN_CH = 1

def differentiable_rollout(
    model: nn.Module,
    geo_t: torch.Tensor,
    norm_history_t: torch.Tensor,
    doy_sin: torch.Tensor,
    doy_cos: torch.Tensor,
    steps: int,
    std_t: torch.Tensor,
    mean_t: torch.Tensor,
    rain_clamp: bool = True,
) -> list:
    B, _, _, H, W = norm_history_t.shape
    dev = norm_history_t.device
    delta = 2.0 * np.pi / 365.0
    cd = torch.cos(torch.tensor(delta, device=dev))
    sd = torch.sin(torch.tensor(delta, device=dev))

    history = norm_history_t.clone()
    sin_v, cos_v = doy_sin.clone(), doy_cos.clone()
    preds_norm = []
    rain_norm_min = (-mean_t[1] / std_t[1]).item()

    for _ in range(steps):
        doy_map = torch.stack([sin_v, cos_v], dim=1)[:, :, None, None].expand(B, 2, H, W)
        geo_b = geo_t.expand(B, -1, -1, -1)
        hist_flat = history.reshape(B, 3 * N_CHANNELS, H, W)
        inp = torch.cat([hist_flat, geo_b, doy_map], dim=1)

        z_next = history[:, -1] + model(inp)
        if rain_clamp:
            z_clamped = z_next.clone()
            z_clamped[:, RAIN_CH] = z_next[:, RAIN_CH].clamp(min=rain_norm_min)
            z_next = z_clamped

        preds_norm.append(z_next)
        history = torch.cat([history[:, 1:], z_next.unsqueeze(1)], dim=1)
        sin_v, cos_v = sin_v * cd + cos_v * sd, cos_v * cd - sin_v * sd

    return preds_norm


class MultiStepHuberLoss(nn.Module):
    def __init__(self, n_channels: int = 6, horizon: int = 10, beta: float = 1.0) -> None:
        super().__init__()
        self.beta = beta
        self.register_buffer("lead_weights", torch.ones(horizon) / horizon)
        self.register_buffer("chan_weights", torch.ones(n_channels) / n_channels)

    def forward(self, preds: list, targets: torch.Tensor) -> torch.Tensor:
        total = torch.tensor(0.0, device=targets.device)
        horizon = min(len(preds), targets.shape[1])
        for k in range(horizon):
            err = F.huber_loss(preds[k], targets[:, k], delta=self.beta, reduction="none")
            step_loss = (self.chan_weights * err.mean(dim=(0, 2, 3))).sum()
            total = total + self.lead_weights[k] * step_loss
        return total


In [ ]:
# ===========================================================================
# Training & Saving Weights
# ===========================================================================
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Loading dataset...")
reader = DatasetReader(NC_PATH)
transform = Transform(STATS_PATH)

std_t  = torch.from_numpy(transform.std).float().to(device)
mean_t = torch.from_numpy(transform.mean).float().to(device)

geo_np = build_geo_features(reader.static[0], reader.static[1], reader.lats, reader.lons)
geo_t  = torch.from_numpy(geo_np).unsqueeze(0).float().to(device)

train_idx = reader.split_indices("train")
val_idx   = reader.split_indices("val")
print(f"Train days: {len(train_idx)} | Val days: {len(val_idx)}")

if MODEL_CLASS == "primary":
    model = GRAST(c_in=C_IN, base_ch=BASE_CH, dropout=DROPOUT).to(device)
    out_path = OUT_DIR / "primary_grast.pt"
else:
    model = FallbackCNN(c_in=C_IN, base_ch=BASE_CH).to(device)
    out_path = OUT_DIR / "fallback_cnn.pt"

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_CLASS} (base_ch={BASE_CH}) | Parameters: {n_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.05)

if CURRICULUM and HORIZON > 1:
    phases = []
    h = 1
    while h < HORIZON:
        phases.append((h, max(EPOCHS // 4, 5)))
        h = min(h * 2, HORIZON)
    rem = EPOCHS - sum(e for _, e in phases)
    phases.append((HORIZON, max(rem, 5)))
else:
    phases = [(HORIZON, EPOCHS)]

log_path = OUT_DIR / "training_log.jsonl"
best_final = {}

for phase_h, phase_epochs in phases:
    print()
    print("=" * 60)
    print(f"Phase: horizon={phase_h}, epochs={phase_epochs}")
    print("=" * 60)
    loss_fn = MultiStepHuberLoss(n_channels=6, horizon=phase_h, beta=1.0).to(device)
    pw_train = WindowIndex(train_idx, n_inputs=3, max_horizon=phase_h)
    pw_val   = WindowIndex(val_idx,   n_inputs=3, max_horizon=phase_h)
    pl_train = DataLoader(ClimateWindowDataset(reader, transform, pw_train, phase_h),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))
    pl_val   = DataLoader(ClimateWindowDataset(reader, transform, pw_val,   phase_h),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(device.type == "cuda"))

    best_val_loss = math.inf
    best_state = None
    best_metrics = {}

    for epoch in range(1, phase_epochs + 1):
        model.train()
        train_loss = 0.0
        t0 = time.time()
        for x_norm, y_norm, doy_sin, doy_cos in pl_train:
            x_norm, y_norm = x_norm.to(device), y_norm.to(device)
            doy_sin, doy_cos = doy_sin.to(device), doy_cos.to(device)
            optimizer.zero_grad()
            preds = differentiable_rollout(model, geo_t, x_norm, doy_sin, doy_cos, phase_h, std_t, mean_t)
            loss = loss_fn(preds, y_norm[:, :phase_h])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        avg_train = train_loss / max(len(pl_train), 1)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_norm, y_norm, doy_sin, doy_cos in pl_val:
                x_norm, y_norm = x_norm.to(device), y_norm.to(device)
                doy_sin, doy_cos = doy_sin.to(device), doy_cos.to(device)
                preds = differentiable_rollout(model, geo_t, x_norm, doy_sin, doy_cos, phase_h, std_t, mean_t)
                val_loss += loss_fn(preds, y_norm[:, :phase_h]).item()
        avg_val = val_loss / max(len(pl_val), 1)
        elapsed = time.time() - t0

        entry = {"phase": f"h{phase_h}", "epoch": epoch, "train_loss": avg_train, "val_loss": avg_val, "elapsed_s": round(elapsed, 1)}
        with open(log_path, "a") as f:
            f.write(json.dumps(entry) + "\n")
        print(f"[h{phase_h}] epoch {epoch:03d}/{phase_epochs:03d}  train={avg_train:.4f}  val={avg_val:.4f}  ({elapsed:.1f}s)")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_metrics = dict(entry)

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    best_final = best_metrics

artifact = {
    "model_name":   f"{MODEL_CLASS}_grast" if MODEL_CLASS == "primary" else "fallback_cnn",
    "model_class":  MODEL_CLASS,
    "c_in":         C_IN,
    "base_ch":      BASE_CH,
    "n_params":     n_params,
    "horizon":      HORIZON,
    "seed":         SEED,
    "epochs":       EPOCHS,
    "best_metrics": best_final,
    "state_dict":   {k: v.cpu() for k, v in model.state_dict().items()},
    "transform":    {"mean": transform.mean.tolist(), "std": transform.std.tolist()},
    "geo_config":   {"n_geo_channels": N_GEO_CHANNELS},
}
torch.save(artifact, out_path)
print(f"Saved checkpoint to: {out_path} ({out_path.stat().st_size / 1e6:.2f} MB)")


In [ ]:
# ===========================================================================
# Verify Saved Checkpoint Loads Cleanly on CPU
# ===========================================================================
ckpt = torch.load(out_path, map_location="cpu", weights_only=False)
test_model = GRAST(c_in=ckpt["c_in"], base_ch=ckpt["base_ch"]) if ckpt["model_class"] == "primary" else FallbackCNN(c_in=ckpt["c_in"], base_ch=ckpt["base_ch"])
test_model.load_state_dict(ckpt["state_dict"])
test_model.eval()
dummy = torch.randn(1, ckpt["c_in"], 32, 36)
with torch.no_grad():
    out = test_model(dummy)
assert out.shape == (1, 6, 32, 36) and torch.isfinite(out).all()
print("Checkpoint verified on CPU! Ready to download:", out_path)
